# Lab №2

## EXECUTION

### 1. Imports and Libraries

In [91]:
# --- CELL 1: PREPARATION OF THE ENVIRONMENT ---
from math import pi
import numpy as np
import roboticstoolbox as rtb
import matplotlib.pyplot as plt
from spatialmath import SE3

print("Libraries imported and environment is ready.")

Libraries imported and environment is ready.


### 2. Loading KUKA KR5 Model

In [ ]:
# --- ROBOT MODEL AND DYNAMIC PARAMETERS ---
# Loading the base model
robot = rtb.models.DH.KR5()

# Applying Lab 1 Mass parameters
robot.links[0].m, robot.links[1].m, robot.links[2].m = 5.0, 6.0, 4.0
robot.links[3].m, robot.links[4].m, robot.links[5].m = 2.0, 2.0, 1.0

# Applying Lab 1 Centers of Mass
robot.links[0].r = [0, 0, 0.1]
robot.links[1].r = [-0.2, 0, 0.1]
robot.links[2].r = [-0.1, 0, 0.05]
robot.links[3].r = [0, 0, 0.02]
robot.links[4].r = [0, 0, 0.02]
robot.links[5].r = [0, 0, 0.01]

# Applying Inertia, Motor inertia, and Friction from Lab 1
for i in range(6):
    robot.links[i].I = [0.05, 0.05, 0.02, 0, 0, 0]
    robot.links[i].Jm = 1e-4
    robot.links[i].B = 0.05
    robot.links[i].Tc = [0.1, -0.1]

print("KUKA KR5 loaded with Lab 1 parameters loaded successfully.")
print(robot)

: 

### 3. Initial Configuration (q_start)

In [ ]:
#---# --- CELL 3: INITIAL CONFIGURATION ---
# Using q_start = [0, 0, 0, 0, 0, 0] from Lab 1
q_start = np.array([0, 0, 0, 0, 0, 0])

print("Initial Configuration (q_start):", q_start)
robot.plot(q_start, block=False)

: 

### 4. Forward Kinematics

In [ ]:
# --- STEP 4: SOLVE FORWARD KINEMATICS ---
T_start = robot.fkine(q_start)
print("Forward Kinematics Solution (Transformation Matrix):")
print(T_start)

: 

### 5. Workspace Construction

In [ ]:
# --- STEP 5a: WORKSPACE CALCULATION ---
n_rings = 15
n_points = 20
q_samples = np.linspace(-pi, pi, n_rings)
q_circle = np.linspace(-pi, pi, n_points)

mesh_data = []
# Triple loop to calculate the reachable coordinates in 3D space
for q1 in q_samples:
    for q2 in np.linspace(-pi/2, pi/2, n_rings):
        ring = []
        for q3 in q_circle:
            # Solving Forward Kinematics to find the point in space
            T = robot.fkine([q1, q2, q3, 0, 0, 0])
            ring.append(T.t)
        mesh_data.append(np.array(ring))

print("Workspace mesh data calculation complete.")

: 

In [ ]:
# --- STEP 5b: WORKSPACE (VISUALIZATION) ---
# Creating the professional high-quality plot for the report.
fig = plt.figure(figsize=(10, 10), dpi=100)
ax = fig.add_subplot(111, projection='3d')

# Drawing the calculated rings to create the wireframe structure
for ring in mesh_data:
    ax.plot(ring[:, 0], ring[:, 1], ring[:, 2], color='red', linewidth=0.5, alpha=0.3)

# Professional styling to match the reference image
ax.set_title("KUKA KR5 Reachable Workspace Analysis", fontsize=14, fontweight='bold')
ax.set_facecolor('#E6E6E6') # Light grey background
ax.set_box_aspect([1,1,1])  # Ensure spherical proportions

# Setting axis limits to center the workspace
limit = 1.0
ax.set_xlim(-limit, limit)
ax.set_ylim(-limit, limit)
ax.set_zlim(-limit, limit)

plt.show()

: 

### 6. Inverse Kinematics

In [ ]:
# --- STEP 6: SOLVING INVERSE KINEMATICS ---
# providing a target location (X, Y, Z) and finding the required joint angles.
from spatialmath import SE3

# Setting the target destination (Position: x=0.4, y=0.2, z=0.3)
# RPY (Roll-Pitch-Yaw) defines the orientation of the robot's hand
T_goal = SE3(0.4, 0.2, 0.3) * SE3.RPY(0, pi, 0)

# Finding the solution using the Levenberg-Marquardt (LM) method
# This numerical solver finds the joint angles needed to reach T_goal
sol = robot.ikine_LM(T_goal)
q_goal = sol.q

print("Target reached! Calculated Joint Angles (q_goal):")
print(q_goal)

# Visualizing the robot at the goal position
robot.plot(q_goal, block=False)

: 

### 7. Trajectory Planning

Method 1 (jtraj)

In [ ]:
# --- STEP 7a: TRAJECTORY PLANNING ---
# Time parameters as specified:
N = 100
t_start = 0
t_stop = 5
t_shag = t_stop / N
time = np.arange(t_start, t_stop, t_shag)

tr_jtraj = rtb.jtraj(q_start, q_goal, time)

print(f"Method 1: jtraj complete with {len(time)} steps over {t_stop} seconds.")

: 

Method 2 (Trapezoidal Velocity)

In [ ]:
# --- STEP 7b: TRAJECTORY PLANNING (METHOD 2: TRAPEZOIDAL) ---
tr_trap = rtb.mtraj(rtb.trapezoidal, q_start, q_goal, time)

print("Method 2: Trapezoidal trajectory (tr_trap) calculated successfully.")

: 

Method 3 (quintic)

In [ ]:
# --- STEP 7c: TRAJECTORY PLANNING (METHOD 3: QUINTIC) ---

tr_quin = rtb.mtraj(rtb.quintic, q_start, q_goal, time)

print("Method 3: Quintic trajectory (tr_quin) calculated.")

: 

### 8. Plotting Graphs

plotting the position graphs for each join

In [ ]:
plt.figure(figsize=(12, 8), dpi=100) 

for g in range(1, 7):
    plt.subplot(2, 3, g)
    # Adjust spacing to ensure titles and labels don't overlap
    plt.subplots_adjust(left=0.1, bottom=0.1, right=0.95, top=0.9, wspace=0.3, hspace=0.4)
    
    # Extracting joint data for each planning method from Lab 1/Lab 2 results
    traektoriya_jtraj = [tr_jtraj.q[i][g-1] for i in range(len(tr_jtraj.q))]
    traektoriya_trap  = [tr_trap.q[i][g-1] for i in range(len(tr_trap.q))]
    traektoriya_quin  = [tr_quin.q[i][g-1] for i in range(len(tr_quin.q))]
    
    # Plotting each method with specific colors and styles
    plt.plot(time, traektoriya_jtraj, linestyle='-', linewidth=2, color=(1,0,0), label="jtraj")
    plt.plot(time, traektoriya_trap, linestyle='-', linewidth=2, color=(0,1,0), label="trapezoidal")
    plt.plot(time, traektoriya_quin, linestyle='--', linewidth=2, color=(0,0,1), label="quintic")
    
    # Labelling the plots according to joint index
    plt.title(f"Joint Position {g}", fontsize=10)
    plt.ylabel(r"$q$ [$rad$]", fontsize=9)
    plt.xlabel(r"$t$ [$sec$]", fontsize=9)
    plt.grid(True)
    plt.legend(fontsize=8)
    
    # Aesthetic styling
    ax = plt.gca()
    ax.set_facecolor((1, 1, 1))
    ax.set_xlim([t_start - 0.1, t_stop + 0.1])

plt.suptitle("Comparison of Joint Posistions for KUKA KR5 Joints", fontsize=14, fontweight='bold')
plt.show()

: 

COMPARATIVE VELOCITY ANALYSIS

In [ ]:

plt.figure(figsize=(12, 8), dpi=100) 

for g in range(1, 7):
    plt.subplot(2, 3, g)
    # Adjust spacing to ensure titles and labels are clear
    plt.subplots_adjust(left=0.1, bottom=0.1, right=0.95, top=0.9, wspace=0.35, hspace=0.4)
    
    # Extracting velocity data (qd) for each planning method
    # We use .qd to get the joint velocities
    traektoriya_jtraj = [tr_jtraj.qd[i][g-1] for i in range(len(tr_jtraj.qd))]
    traektoriya_trap  = [tr_trap.qd[i][g-1] for i in range(len(tr_trap.qd))]
    traektoriya_quin  = [tr_quin.qd[i][g-1] for i in range(len(tr_quin.qd))]
    
    # Plotting each method with specific colors and styles
    plt.plot(time, traektoriya_jtraj, linestyle='-', linewidth=2, color=(1,0,0), label="jtraj")
    plt.plot(time, traektoriya_trap, linestyle='-', linewidth=2, color=(0,1,0), label="trapezoidal")
    plt.plot(time, traektoriya_quin, linestyle='--', linewidth=2, color=(0,0,1), label="quintic")
    
    # Setting the titles and labels in English/Standard format
    plt.title(f"Joint Velocity {g}", fontsize=10)
    plt.ylabel(r"$\dot{q}$ [$rad/sec$]", fontsize=9)
    plt.xlabel(r"$t$ [$sec$]", fontsize=9)
    plt.grid(True)
    plt.legend(fontsize=8)
    
    # Aesthetic styling
    ax = plt.gca()
    ax.set_facecolor((1, 1, 1))
    ax.set_xlim([t_start - 0.1, t_stop + 0.1])

plt.suptitle("Comparison of Joint Velocities for KUKA KR5", fontsize=14, fontweight='bold')
plt.show()

: 

Comparative Acceleration Analysis

In [ ]:
# --- STEP 8c: COMPARATIVE ACCELERATION ANALYSIS ---

plt.figure(figsize=(12, 8), dpi=100)

for g in range(1, 7):
    plt.subplot(2, 3, g)
    # Adjusting intervals so titles and labels don't overlap
    plt.subplots_adjust(left=0.1, bottom=0.1, right=0.95, top=0.9, wspace=0.35, hspace=0.4)
    
    # Extracting acceleration data (qdd) for each planning method
    traektoriya_jtraj = [tr_jtraj.qdd[i][g-1] for i in range(len(tr_jtraj.qdd))]
    traektoriya_trap  = [tr_trap.qdd[i][g-1] for i in range(len(tr_trap.qdd))]
    traektoriya_quin  = [tr_quin.qdd[i][g-1] for i in range(len(tr_quin.qdd))]
    
    # Plotting acceleration with consistent styles
    plt.plot(time, traektoriya_jtraj, linestyle='-', linewidth=2, color=(1,0,0), label="jtraj")
    plt.plot(time, traektoriya_trap, linestyle='-', linewidth=2, color=(0,1,0), label="trapezoidal")
    plt.plot(time, traektoriya_quin, linestyle='--', linewidth=2, color=(0,0,1), label="quintic")
    
    # Labels and Titles
    plt.title(f"Joint Acceleration {g}", fontsize=10)
    plt.ylabel(r"$\ddot{q}$ [$rad/sec^2$]", fontsize=9)
    plt.xlabel(r"$t$ [$sec$]", fontsize=9)
    plt.grid(True)
    plt.legend(fontsize=8)
    
    # Plot limits and background
    ax = plt.gca()
    ax.set_facecolor((1,1,1))
    ax.set_xlim([t_start-0.1, t_stop+0.1])

plt.suptitle("Comparison of Joint Accelerations for KUKA KR5", fontsize=14, fontweight='bold')
plt.show()

: 

### 9.LAB TWO (2) REPORT’S SUMMARY

LAB TWO (2) REPORT’S SUMMARY

Part 1. Cell Execution Summary

Cell 1-3 (Setup): Defined the DH Parameters and initialized the KUKA KR5 robot model based on Lab 1 specifications
Cell 4-6 (Inverse Kinematics): Calculated the starting and ending joint configurations (q_start and q_goal) for the desired workspace coordinates
Cell 7a (jtraj): Used the standard toolbox trajectory function, which internally generates a smooth 5th-order polynomial path.
Cell 7b (trapezoidal): Implemented a Trapezoidal Velocity profile, which maintains a constant velocity during the middle of the motion
Cell 7c (quintic): Applied the Quintic method within the mtraj framework to provide an alternative high-order polynomial path
Cell 8a (Position Plotting): Visualized the displacement of all 6 joints over 5 seconds to verify they reached their targets.
Cell 8b (Velocity Plotting): Compared how fast each joint moved across the three different planning methods
Cell 8c (Acceleration Plotting): Analyzed the "jerkiness" or smoothness of the motion by plotting the second derivative of the position.

Part 2: Technical Analysis

1. Position Analysis (q vs t)

All three methods: jtraj, trapezoidal, and quintic successfully moved the robot from the initial state to the final state within the allocated 5 seconds. The curves for jtraj and quintic appear almost identical, showing a very natural "S-curve" shape. The trapezoidal method, however, shows a more linear progression in the middle section of the graph.

2. Velocity Analysis (q˙ vs t)

jtraj & quintic: These methods produce a smooth, bell-shaped velocity curve. The velocity starts at zero, peaks gradually, and returns to zero softly.
Trapezoidal: This method shows a distinct "flat top" or plateau. This indicates that the robot reaches a maximum set velocity and holds it for a period before decelerating.

3. Acceleration Analysis (q¨ vs t)

This is the most critical area for robot health.
jtraj/quintic: The acceleration changes continuously and smoothly. This minimizes mechanical "jerk," which reduces wear and tear on the KUKA KR5's motors and gearboxes.
Trapezoidal: The acceleration graph shows "step" functions (sudden jumps). These instantaneous changes in acceleration represent high-impact forces that can cause the robot to vibrate or shake in a real-world setting.

Part 3: Conclusion

The simulation successfully demonstrated three distinct ways to move the KUKA KR5 between two points in space.

Recommendation: For precision tasks such as painting or arc welding with the KUKA KR5, the jtraj or quintic methods are superior because they provide the smoothest motion.
Efficiency: The trapezoidal method is useful when the robot needs to maintain a specific constant speed for a task (like a conveyor belt synchronization) but requires better smoothing at the start and end points.
Overall, the project confirms that a time-step of N=100 over 5 seconds provides sufficient resolution for high-quality trajectory analysis.

